# MML Ch 7 — Continuous Optimization

**Source:** Mathematics for Machine Learning (Deisenroth, Faisal, Ong), Chapter 7  
**Notes:** `study/ml-math/notes/continuous-optimization.md`

This notebook implements and visualizes the core optimization algorithms and concepts
from Chapter 7: gradient descent variants, constrained optimization via Lagrange
multipliers, convexity, Newton's method, and the implicit regularization of SGD.

---

## Contents
1. Gradient Descent variants — vanilla GD, Momentum, Adam from scratch
2. Learning rate effect — convergence vs. divergence
3. Constrained optimization — Lagrange multipliers and KKT
4. Convexity check — Jensen's inequality for exp(x)
5. Newton's method vs. GD — convergence comparison
6. SGD noise as regularization — overparameterized polynomial fitting

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LogNorm

plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})
rng = np.random.default_rng(0)

---
## 1. Gradient Descent Variants from Scratch

**Objective:** minimize $f(x, y) = x^2 + 10y^2$ — an ill-conditioned quadratic.

The condition number is $\kappa = 10/1 = 10$, meaning the surface is 10× steeper
in the $y$-direction. This makes plain GD bounce (zig-zag), while momentum and Adam
navigate it more efficiently.

**Update rules:**
- **Vanilla GD:** $x_{t+1} = x_t - \alpha \nabla f(x_t)$
- **Momentum:** $v_t = \beta v_{t-1} + \nabla f(x_t)$; $x_{t+1} = x_t - \alpha v_t$
- **Adam:** exponential moving averages of $g_t$ and $g_t^2$, with bias correction

In [ ]:
def f(xy):        return xy[0]**2 + 10 * xy[1]**2
def grad_f(xy):   return np.array([2*xy[0], 20*xy[1]])

def run_gd(x0, lr=0.05, n_steps=150):
    """Vanilla gradient descent."""
    x = x0.copy(); path = [x.copy()]
    for _ in range(n_steps):
        x = x - lr * grad_f(x)
        path.append(x.copy())
    return np.array(path)

def run_momentum(x0, lr=0.05, beta=0.85, n_steps=150):
    """GD with momentum."""
    x = x0.copy(); v = np.zeros(2); path = [x.copy()]
    for _ in range(n_steps):
        v = beta * v + grad_f(x)
        x = x - lr * v
        path.append(x.copy())
    return np.array(path)

def run_adam(x0, lr=0.1, beta1=0.9, beta2=0.999, eps=1e-8, n_steps=150):
    """Adam optimizer."""
    x = x0.copy(); m = np.zeros(2); v = np.zeros(2)
    path = [x.copy()]
    for t in range(1, n_steps + 1):
        g = grad_f(x)
        m = beta1 * m + (1 - beta1) * g
        v = beta2 * v + (1 - beta2) * g**2
        m_hat = m / (1 - beta1**t)
        v_hat = v / (1 - beta2**t)
        x = x - lr * m_hat / (np.sqrt(v_hat) + eps)
        path.append(x.copy())
    return np.array(path)

x0 = np.array([-3.0, 2.0])
path_gd  = run_gd(x0)
path_mom = run_momentum(x0)
path_adam = run_adam(x0)

# ---- Contour background ----
x_range = np.linspace(-4, 4, 300)
y_range = np.linspace(-2.5, 2.5, 300)
X, Y = np.meshgrid(x_range, y_range)
Z = X**2 + 10 * Y**2

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: trajectory plot
ax = axes[0]
cs = ax.contour(X, Y, Z, levels=np.logspace(-1, 1.5, 20), cmap='Blues', alpha=0.5)
plt.colorbar(cs, ax=ax, label='f(x,y)')

colors = {'GD': 'steelblue', 'Momentum': 'darkorange', 'Adam': 'seagreen'}
for path, label, color in [(path_gd, 'GD', colors['GD']),
                            (path_mom, 'Momentum', colors['Momentum']),
                            (path_adam, 'Adam', colors['Adam'])]:
    ax.plot(path[:, 0], path[:, 1], '-', color=color, lw=1.5, alpha=0.85, label=label)
    ax.plot(*path[0],  'o', color=color, ms=8)
    ax.plot(*path[-1], '*', color=color, ms=12)

ax.plot(0, 0, 'k+', ms=14, mew=2.5, label='Minimum (0,0)')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('f(x,y) = x² + 10y²\nOptimizer trajectories')
ax.legend(fontsize=9, loc='lower right')

# Right: convergence curves
ax = axes[1]
for path, label, color in [(path_gd,  'GD',       colors['GD']),
                            (path_mom, 'Momentum', colors['Momentum']),
                            (path_adam,'Adam',      colors['Adam'])]:
    losses = [f(p) for p in path]
    ax.semilogy(losses, color=color, lw=2, label=label)

ax.set_xlabel('Step'); ax.set_ylabel('f(x, y)  [log scale]')
ax.set_title('Convergence curves')
ax.legend(fontsize=10)

fig.suptitle('Section 7.1 — Gradient Descent Variants', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Final loss — GD: {f(path_gd[-1]):.2e},  "
      f"Momentum: {f(path_mom[-1]):.2e},  "
      f"Adam: {f(path_adam[-1]):.2e}")

---
## 2. Learning Rate Effect

The learning rate $\alpha$ is the most sensitive hyperparameter in gradient descent:

- **Too small ($\alpha = 0.01$):** converges, but very slowly
- **Moderate ($\alpha = 0.1$):** good convergence
- **Too large ($\alpha = 0.9$):** overshoots the minimum and diverges

For a quadratic $f(x) = ax^2$, GD is stable iff $\alpha < 1/a$.
Here with $a=10$ (in the $y$ direction), divergence occurs for $\alpha > 0.1$.

In [ ]:
# Use 1D slice: g(y) = 10y² for clarity of learning-rate effect
def g(y):       return 10 * y**2
def grad_g(y):  return 20 * y

y0 = 2.0
lrs = [0.01, 0.09, 0.11]  # sub-critical, near-critical, super-critical
labels = ['lr = 0.01  (slow)', 'lr = 0.09  (converges)', 'lr = 0.11  (diverges)']
colors_lr = ['steelblue', 'seagreen', 'crimson']
n_steps = 40

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax = axes[0]
for lr, lbl, col in zip(lrs, labels, colors_lr):
    y = y0
    path = [y]
    for _ in range(n_steps):
        y = y - lr * grad_g(y)
        if abs(y) > 50:  # stop if diverging
            path.append(y)
            break
        path.append(y)
    t_axis = np.arange(len(path))
    ax.plot(t_axis, path, '-o', color=col, lw=2, ms=4, label=lbl)

ax.set_xlabel('Step'); ax.set_ylabel('y')
ax.set_title('1D slice: g(y) = 10y²\nLearning-rate sensitivity')
ax.set_ylim(-5, 20)
ax.axhline(0, color='k', lw=1, ls='--', alpha=0.5)
ax.legend(fontsize=9)

ax = axes[1]
for lr, lbl, col in zip(lrs, labels, colors_lr):
    y = y0
    losses = [g(y)]
    for _ in range(n_steps):
        y = y - lr * grad_g(y)
        loss = g(y)
        if abs(loss) > 1e6:
            losses.append(loss)
            break
        losses.append(loss)
    ax.semilogy(losses, '-', color=col, lw=2, label=lbl)

ax.set_xlabel('Step'); ax.set_ylabel('g(y)  [log scale]')
ax.set_title('Loss convergence by learning rate')
ax.legend(fontsize=9)

fig.suptitle('Section 7.1 — Learning Rate Effect', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("Stability condition for g(y) = 10y²: α < 1/10 = 0.1")
print("GD update: y ← y − α·20y = y(1 − 20α)")
print("Stable iff |1 − 20α| < 1  ⟺  0 < α < 0.1")

---
## 3. Constrained Optimization — Lagrange Multipliers and KKT

**Problem:** Maximize $f(x, y) = xy$ subject to $x + y = 1$.

**Lagrangian:** $L(x, y, \lambda) = xy - \lambda(x + y - 1)$

**KKT stationarity conditions:**
$$\frac{\partial L}{\partial x} = y - \lambda = 0 \implies y = \lambda$$
$$\frac{\partial L}{\partial y} = x - \lambda = 0 \implies x = \lambda$$
$$\frac{\partial L}{\partial \lambda} = -(x + y - 1) = 0 \implies x + y = 1$$

Solution: $x^* = y^* = 1/2$, $\lambda^* = 1/2$, $f^* = 1/4$.

In [ ]:
# --- Analytical solution ---
x_star, y_star = 0.5, 0.5
lam_star = 0.5
f_star = x_star * y_star

print("=== Constrained Optimization: max xy s.t. x+y=1 ===")
print(f"  Analytical solution: x* = {x_star},  y* = {y_star}")
print(f"  Lagrange multiplier: λ* = {lam_star}")
print(f"  Optimal value:       f* = {f_star}")

# --- Numerical verification: sweep along the constraint ---
t = np.linspace(-1, 2, 400)   # x = t, y = 1 - t
f_on_constraint = t * (1 - t)

idx_max = np.argmax(f_on_constraint)
print(f"\n  Numerical max on constraint:")
print(f"  x = {t[idx_max]:.4f},  y = {1-t[idx_max]:.4f},  f = {f_on_constraint[idx_max]:.6f}")

# --- KKT residual check ---
dL_dx = y_star - lam_star   # should be 0
dL_dy = x_star - lam_star   # should be 0
constraint = x_star + y_star - 1  # should be 0
print(f"\n  KKT residuals: ∂L/∂x = {dL_dx},  ∂L/∂y = {dL_dy},  constraint = {constraint}")

# --- Visualization ---
x_grid = np.linspace(-1, 2, 300)
y_grid = np.linspace(-1, 2, 300)
Xg, Yg = np.meshgrid(x_grid, y_grid)
Fg = Xg * Yg

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: contour map with constraint line
ax = axes[0]
levels = np.linspace(-1, 0.6, 20)
cs = ax.contourf(Xg, Yg, Fg, levels=levels, cmap='RdYlGn', alpha=0.75)
plt.colorbar(cs, ax=ax, label='f(x,y) = xy')

# Constraint line x + y = 1
x_line = np.linspace(-0.5, 1.5, 200)
ax.plot(x_line, 1 - x_line, 'k-', lw=2.5, label='constraint x+y=1')

# Optimal point
ax.plot(x_star, y_star, 'r*', ms=16, label=f'KKT solution ({x_star},{y_star})')

# Gradient of f at optimum and gradient of constraint
ax.annotate('', xy=(x_star + 0.2, y_star),
            xytext=(x_star, y_star),
            arrowprops=dict(arrowstyle='->', color='darkblue', lw=2))
ax.annotate('∇f', xy=(x_star + 0.22, y_star), color='darkblue', fontsize=10)
ax.annotate('', xy=(x_star + 0.1, y_star + 0.1),
            xytext=(x_star, y_star),
            arrowprops=dict(arrowstyle='->', color='darkred', lw=2))
ax.annotate('λ∇g', xy=(x_star + 0.12, y_star + 0.12), color='darkred', fontsize=10)

ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('max xy  s.t.  x+y=1\nContour map with constraint')
ax.legend(fontsize=9)
ax.set_xlim(-0.5, 1.5); ax.set_ylim(-0.5, 1.5)
ax.set_aspect('equal')

# Right: f along the constraint
ax = axes[1]
ax.plot(t, f_on_constraint, 'steelblue', lw=2.5, label='f(t, 1−t) = t(1−t)')
ax.axvline(0.5, color='crimson', ls='--', lw=1.5, label='x* = 0.5')
ax.axhline(0.25, color='darkorange', ls=':', lw=1.5, label='f* = 0.25')
ax.plot(0.5, 0.25, 'r*', ms=14)
ax.set_xlabel('x  (with y = 1−x on constraint)')
ax.set_ylabel('f(x, y) = xy')
ax.set_title('Objective along constraint')
ax.legend(fontsize=10)

fig.suptitle('Section 7.2 — Lagrange Multipliers / KKT Conditions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Convexity — Jensen's Inequality for exp(x)

For a convex function $f$, **Jensen's inequality** states:

$$f(\mathbb{E}[X]) \le \mathbb{E}[f(X)]$$

"The function of the average is at most the average of the function."

**Verification for $f(x) = e^x$:**  
Since $f''(x) = e^x > 0$ everywhere, $f$ is strictly convex, so Jensen's inequality
holds with strict inequality (for non-constant $X$).

**Importance in ML:** Jensen's inequality is used to derive the ELBO in variational
inference, prove KL divergence $\ge 0$, and analyze the EM algorithm.

In [ ]:
# Numerical verification of Jensen's inequality for f(x) = exp(x)
n_trials = 10
print("=== Jensen's Inequality: f(E[X]) ≤ E[f(X)] for f(x)=exp(x) ===")
print(f"{'n_samples':>10}  {'E[X]':>8}  {'f(E[X])':>10}  {'E[f(X)]':>10}  {'Holds?':>8}")
print("-" * 55)
for n in [10, 100, 1000, 10000, 100000]:
    X = rng.uniform(-2, 2, n)
    ex = np.mean(X)
    f_of_ex = np.exp(ex)
    ef_of_x = np.mean(np.exp(X))
    holds = f_of_ex <= ef_of_x + 1e-10
    print(f"{n:>10,}  {ex:>8.4f}  {f_of_ex:>10.4f}  {ef_of_x:>10.4f}  {'✓' if holds else '✗':>8}")

# Visual proof for a single distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Left: geometric picture of Jensen's inequality
ax = axes[0]
x_range = np.linspace(-3, 3, 300)
ax.plot(x_range, np.exp(x_range), 'steelblue', lw=2.5, label='f(x) = eˣ  (convex)')

# Two sample points
a, b, theta = -1.5, 1.5, 0.4
x_mid = theta * a + (1 - theta) * b   # convex combination
y_mid_chord = theta * np.exp(a) + (1 - theta) * np.exp(b)  # chord value
y_mid_curve = np.exp(x_mid)            # curve value

ax.plot([a, b], [np.exp(a), np.exp(b)], 'crimson', lw=2, ls='--', label='chord (convex combination)')
ax.plot(x_mid, y_mid_chord, 'r^', ms=10, label=f'θf(a)+(1-θ)f(b) = {y_mid_chord:.3f}  [chord]')
ax.plot(x_mid, y_mid_curve, 'gs', ms=10, label=f'f(θa+(1-θ)b) = {y_mid_curve:.3f}  [curve]')
ax.annotate('', xy=(x_mid, y_mid_chord), xytext=(x_mid, y_mid_curve),
            arrowprops=dict(arrowstyle='<->', color='purple', lw=2))
ax.text(x_mid + 0.15, (y_mid_chord + y_mid_curve)/2,
        f'gap = {y_mid_chord - y_mid_curve:.3f}', color='purple', fontsize=9)
ax.plot([a], [np.exp(a)], 'ko', ms=8)
ax.plot([b], [np.exp(b)], 'ko', ms=8)
ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.set_title('Jensen\'s inequality: f(convex combo) ≤ convex combo of f')
ax.legend(fontsize=8, loc='upper left')
ax.set_ylim(0, 6)

# Right: gap E[f(X)] − f(E[X]) grows with variance of X
ax = axes[1]
sigmas = np.linspace(0.01, 3, 50)
gaps = []
for sig in sigmas:
    X_s = rng.normal(0, sig, 100_000)
    gap = np.mean(np.exp(X_s)) - np.exp(np.mean(X_s))
    gaps.append(gap)
# Theoretical: for X~N(0,σ²), E[exp(X)] = exp(σ²/2), E[X]=0, f(E[X])=1
# Gap = exp(σ²/2) - 1
theoretical_gaps = np.exp(sigmas**2 / 2) - 1
ax.plot(sigmas, theoretical_gaps, 'crimson', lw=2.5, label='Theory: exp(σ²/2) - 1')
ax.plot(sigmas, gaps, 'steelblue', lw=1.5, ls='--', alpha=0.8, label='Empirical')
ax.set_xlabel('σ (std of X ~ N(0, σ²))')
ax.set_ylabel('E[eˣ] − e^{E[X]}')
ax.set_title('Jensen gap grows with variance\n(X ~ N(0, σ²))')
ax.legend(fontsize=10)

fig.suptitle('Section 7.3 — Convexity and Jensen\'s Inequality', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 5. Newton's Method vs. Gradient Descent

**Newton's method** uses second-order information (the Hessian) to take a smarter step:

$$x_{t+1} = x_t - [\nabla^2 f(x_t)]^{-1} \nabla f(x_t)$$

For a strongly convex quadratic, Newton's method converges in **one step** (exact).
For general smooth strongly convex functions, it has **quadratic convergence**
(error squared each iteration), while GD has only linear convergence.

**Tradeoff:** Newton needs $O(d^3)$ per step to invert the Hessian vs $O(d)$ for GD.
This makes Newton impractical for large $d$ (e.g., neural networks with millions of parameters).

In [ ]:
# Strongly convex function: f(x) = x^4 + 3x^2 - 2x + 1
# (non-quadratic, so Newton won't converge in 1 step, but much faster than GD)
def h(x):       return x**4 + 3*x**2 - 2*x + 1
def grad_h(x):  return 4*x**3 + 6*x - 2
def hess_h(x):  return 12*x**2 + 6         # scalar Hessian

def run_gd_1d(x0, lr=0.05, tol=1e-10, max_steps=500):
    x = x0; path = [x]
    for _ in range(max_steps):
        x = x - lr * grad_h(x)
        path.append(x)
        if abs(grad_h(x)) < tol:
            break
    return np.array(path)

def run_newton_1d(x0, tol=1e-10, max_steps=50):
    x = x0; path = [x]
    for _ in range(max_steps):
        x = x - grad_h(x) / hess_h(x)
        path.append(x)
        if abs(grad_h(x)) < tol:
            break
    return np.array(path)

x0_1d = 3.0
path_gd_1d  = run_gd_1d(x0_1d)
path_newton = run_newton_1d(x0_1d)

# True minimum (numerical)
from scipy.optimize import minimize_scalar
x_opt = minimize_scalar(h).x

gd_errors     = np.abs(np.array([h(x) for x in path_gd_1d])     - h(x_opt))
newton_errors = np.abs(np.array([h(x) for x in path_newton]) - h(x_opt))

print(f"=== f(x) = x⁴ + 3x² − 2x + 1,  x₀ = {x0_1d} ===")
print(f"  True minimum: x* ≈ {x_opt:.6f},  f* ≈ {h(x_opt):.6f}")
print(f"  GD steps to ε=1e-6:      {np.argmax(gd_errors < 1e-6) if any(gd_errors < 1e-6) else '>500'}")
print(f"  Newton steps to ε=1e-6:  {np.argmax(newton_errors < 1e-6) if any(newton_errors < 1e-6) else '>50'}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Left: function and paths
ax = axes[0]
x_plot = np.linspace(-1, 4, 400)
ax.plot(x_plot, h(x_plot), 'gray', lw=2, alpha=0.6, label='f(x)')
# Mark GD path on curve
show_steps = min(20, len(path_gd_1d))
ax.plot(path_gd_1d[:show_steps], [h(x) for x in path_gd_1d[:show_steps]],
        'o-', color='steelblue', ms=5, lw=1.5, alpha=0.7, label=f'GD ({len(path_gd_1d)} steps)')
show_newton = len(path_newton)
ax.plot(path_newton, [h(x) for x in path_newton],
        's-', color='crimson', ms=7, lw=2, label=f'Newton ({show_newton} steps)')
ax.plot(x_opt, h(x_opt), 'k*', ms=14, label='minimum x*')
ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.set_title('Newton vs. GD: f(x) = x⁴+3x²−2x+1')
ax.legend(fontsize=9)
ax.set_xlim(-0.5, 3.5)

# Right: convergence error
ax = axes[1]
eps_floor = 1e-15
ax.semilogy(np.maximum(gd_errors, eps_floor), 'steelblue', lw=2,
            label=f'GD  (lr=0.05, {len(path_gd_1d)} steps)')
ax.semilogy(np.maximum(newton_errors, eps_floor), 'crimson', lw=2, ls='--',
            label=f'Newton ({show_newton} steps)')
ax.set_xlabel('Step')
ax.set_ylabel('|f(xₜ) − f*|  [log]')
ax.set_title('Convergence: GD (linear) vs. Newton (quadratic)')
ax.legend(fontsize=9)
ax.set_xlim(0, min(80, len(path_gd_1d)))

fig.suptitle('Section 7.1 — Newton\'s Method vs. Gradient Descent', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 6. SGD Noise as Implicit Regularization

**Setup:** Fit an overparameterized polynomial (degree 10) to 15 clean data points
sampled from a quadratic $y = 0.5x^2 - x + 1$.

Both batch GD and SGD can fit the training data perfectly (zero loss),
but they may converge to **different solutions** in the underdetermined regime.

**Key claim from the notes (§7.1):** SGD's gradient noise tends to find *flatter*
minima — solutions that are simpler and generalize better. This is an implicit
regularization effect of using mini-batches.

We demonstrate this by:
1. Fitting with batch GD (full dataset every step)
2. Fitting with SGD (one sample at a time)

Measuring solution complexity via coefficient norms $\|\theta\|_2$.

In [ ]:
# Data from y = 0.5x² - x + 1 + small noise
x_data = np.linspace(-2, 3, 15)
y_true = 0.5 * x_data**2 - x_data + 1
y_data = y_true + rng.normal(0, 0.05, len(x_data))  # tiny noise

degree = 10  # overparameterized (15 points, 11 coefficients)

# Design matrix with polynomial features (normalized for stability)
x_mean, x_std = x_data.mean(), x_data.std()
x_norm = (x_data - x_mean) / x_std

def design_matrix(x, deg):
    return np.column_stack([x**d for d in range(deg + 1)])

Phi = design_matrix(x_norm, degree)  # shape (15, 11)

def poly_predict(theta, x):
    return design_matrix(x, degree) @ theta

def mse_loss(theta, X_feat, y_tgt):
    residuals = X_feat @ theta - y_tgt
    return 0.5 * np.mean(residuals**2)

def grad_mse(theta, X_feat, y_tgt):
    residuals = X_feat @ theta - y_tgt
    return X_feat.T @ residuals / len(y_tgt)

n_epochs = 3000
lr_gd  = 0.02
lr_sgd = 0.005

# --- Batch GD ---
theta_gd = rng.normal(0, 0.01, degree + 1)
losses_gd = []
for _ in range(n_epochs):
    g = grad_mse(theta_gd, Phi, y_data)
    theta_gd = theta_gd - lr_gd * g
    losses_gd.append(mse_loss(theta_gd, Phi, y_data))

# --- SGD (same init for fairness) ---
rng_sgd = np.random.default_rng(1)  # separate seed
theta_sgd = rng_sgd.normal(0, 0.01, degree + 1)
losses_sgd = []
n_train = len(x_data)
for epoch in range(n_epochs):
    idx = rng_sgd.integers(0, n_train)  # one random sample
    x_i = Phi[idx:idx+1]  # (1, D)
    y_i = y_data[idx:idx+1]
    g = grad_mse(theta_sgd, x_i, y_i)
    theta_sgd = theta_sgd - lr_sgd * g
    if epoch % 10 == 0:
        losses_sgd.append(mse_loss(theta_sgd, Phi, y_data))

# --- Evaluate ---
x_dense_norm = np.linspace(x_norm.min() - 0.3, x_norm.max() + 0.3, 300)
y_pred_gd  = poly_predict(theta_gd,  x_dense_norm)
y_pred_sgd = poly_predict(theta_sgd, x_dense_norm)

# True function on same x (un-normalize for display)
x_dense = x_dense_norm * x_std + x_mean
y_true_dense = 0.5 * x_dense**2 - x_dense + 1

norm_gd  = np.linalg.norm(theta_gd)
norm_sgd = np.linalg.norm(theta_sgd)

print(f"=== Overparameterized Polynomial (degree {degree}) ===")
print(f"  GD  — final loss: {losses_gd[-1]:.4e},  ‖θ‖₂ = {norm_gd:.3f}")
print(f"  SGD — final loss: {mse_loss(theta_sgd, Phi, y_data):.4e},  ‖θ‖₂ = {norm_sgd:.3f}")
print(f"  SGD finds a {'simpler' if norm_sgd < norm_gd else 'more complex'} solution "
      f"(smaller ‖θ‖ = {'yes' if norm_sgd < norm_gd else 'no'})")

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

# Left: GD fit
ax = axes[0]
ax.scatter(x_data, y_data, color='k', s=40, zorder=5, label='Training data')
ax.plot(x_dense, y_true_dense, 'gray', lw=2, ls='--', label='True y = 0.5x²−x+1', alpha=0.7)
clip_lo, clip_hi = -3, 5
y_gd_clipped = np.clip(y_pred_gd, clip_lo, clip_hi)
ax.plot(x_dense, y_gd_clipped, 'steelblue', lw=2.5, label=f'GD fit  ‖θ‖={norm_gd:.1f}')
ax.set_xlabel('x (normalized)'); ax.set_ylabel('y')
ax.set_title('Batch GD')
ax.legend(fontsize=8); ax.set_ylim(clip_lo, clip_hi)

# Middle: SGD fit
ax = axes[1]
ax.scatter(x_data, y_data, color='k', s=40, zorder=5, label='Training data')
ax.plot(x_dense, y_true_dense, 'gray', lw=2, ls='--', label='True y = 0.5x²−x+1', alpha=0.7)
y_sgd_clipped = np.clip(y_pred_sgd, clip_lo, clip_hi)
ax.plot(x_dense, y_sgd_clipped, 'crimson', lw=2.5, label=f'SGD fit  ‖θ‖={norm_sgd:.1f}')
ax.set_xlabel('x (normalized)'); ax.set_ylabel('y')
ax.set_title('SGD (1 sample/step)')
ax.legend(fontsize=8); ax.set_ylim(clip_lo, clip_hi)

# Right: convergence + coefficient norms
ax = axes[2]
ax.semilogy(losses_gd, 'steelblue', lw=2, label='Batch GD loss')
step_range_sgd = np.arange(len(losses_sgd)) * 10
ax.semilogy(step_range_sgd, losses_sgd, 'crimson', lw=2, alpha=0.8, label='SGD loss')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE loss  [log]')
ax.set_title(f'Convergence\n‖θ‖: GD={norm_gd:.2f}, SGD={norm_sgd:.2f}')
ax.legend(fontsize=9)

fig.suptitle('Section 7.1 — SGD Noise as Implicit Regularization',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Summary

| Section | Key insight |
|---------|-------------|
| GD variants | Momentum smooths zig-zagging; Adam adapts per-parameter learning rates via 1st and 2nd moment estimates |
| Learning rate | Stable iff $\alpha < 1/L$ (L = Lipschitz constant of gradient); too large → divergence |
| Lagrange / KKT | At optimum, $\nabla f \parallel \nabla g$; complementary slackness: $\mu_j h_j = 0$ |
| Jensen's inequality | $f(\mathbb{E}[X]) \le \mathbb{E}[f(X)]$ for convex $f$; gap grows with variance of $X$ |
| Newton vs. GD | Newton: quadratic convergence, $O(d^3)$/step; GD: linear convergence, $O(d)$/step |
| SGD regularization | Gradient noise biases toward simpler (smaller-norm) solutions — implicit regularization |